## **_Sample Modeling_** (for Reinforcement Learning)

* 이 페이지는 "Drying Oven"의 가상 센서 데이터를 이용하여 PPO 모델을 교육하고 예측하는 예제를 구현합니다.
* PPO는 기본적으로 Linear 모델을 지원합니다.
* pip install numpy pandas tensor ipykernel torch scikit-learn gymnasium stable_baselines3

In [2]:
# ----------------------------------------------------
# Global 변수 선언
# ----------------------------------------------------
import os
import sys
from pathlib import Path
from datetime import datetime

csv_file_pathname = f".{os.sep}data{os.sep}drying_oven_sensor_v1.1.csv"
model_path_name = f".{os.sep}model{os.sep}ppo_dryer_control_v1.1.pth"
model_path_name = lambda: f".{os.sep}model{os.sep}ppo_dryer_control_v1.1_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pth"
global_num_samples = 50000
parent_dir = os.path.dirname(os.getcwd())
project_home_dir = str(Path.cwd().parent.parent)
venv_name = "samplemodelingv1.1"

reward_gen_fn="lib/reward/dryer_oven/gen_reward_v1.py"
reward_gen_fn = reward_gen_fn.replace("/", os.sep)
reward_gen_fn_pythonfile = project_home_dir + os.sep + reward_gen_fn

In [3]:
# ----------------------------------------------------
# 개발 환경 초기화
# 한 번만 호출하여 주십시요.


# 1.1 set pythonpath to import from parent directory
def _init_environ():
    python_path = os.environ.get("PYTHONPATH")
    if python_path is None:
        python_path = ""
    sys.path.append(parent_dir)
    sys.path.append(project_home_dir)
    python_lib_path = [
        f'{project_home_dir}{os.sep}{venv_name}{os.sep}{"Lib" if sys.platform == "win32" else "lib"}{os.sep}site-packages',
        project_home_dir,
        python_path
    ]
    os.environ["PYTHONPATH"] = ";".join(python_lib_path)

    os.environ["CLEARML_WEB_HOST"] = "http://172.16.8.168:8080/"
    os.environ["CLEARML_API_HOST"] = "http://172.16.8.168:8008/"
    os.environ["CLEARML_FILES_HOST"] = "http://172.16.8.168:8081"
    os.environ["CLEARML_API_ACCESS_KEY"] = "SIJ8V8YP9PL25YEAVAWGP9NV11TMOQ"
    os.environ["CLEARML_API_SECRET_KEY"] = "6x_MEHvgrUv9-TFFDrE9BE7vb3JYOXDWq4kQtqbd58nQ5pqHEGMt6qQXxF7_HCyzq7E"

_init_environ()


In [8]:
!echo $PYTHONPATH
from lib.simulate.genetate_data import generate_data_v1

# 1. 설정값 초기화
num_samples = global_num_samples
target_temp = 80
initial_temp = 25
drop_envs = [(0.4, 100, 30), (0.5, 200, 40)]
spike_envs = [(0.6, 300, 30), (0.7, 200, 40), (0.8, 200, 70)]
verbose = 0


generate_data_v1(drop_envs=drop_envs,
                 spike_envs=spike_envs,
                 csv_file_pathname=csv_file_pathname,
                 num_samples=global_num_samples,
                 target_temp=target_temp,
                 initial_temp=initial_temp,
                 verbose=verbose)

/home/simpson/work/mlops_jupyter_notebooks/samplemodelingv1.1/lib/site-packages;/home/simpson/work/mlops_jupyter_notebooks;


Start to generate sample data. [2026-03-29 21:20:16.947492]
End to generate sample data. [2026-03-29 21:20:17.256975]


#### Step 2 Env 생성

In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class DryerEnv(gym.Env):
    """
    RecurrentPPO 학습을 위한 건조기 온도 제어 환경
    목표: 팬 RPM을 조절하여 내부 온도를 80도로 유지하기
    """
    init_sim_data = [{
            "timestamp": 0.0,
            "inner_temp": 79.5,  # 실측 온도
            "error": -0.5,       # 실측 온도 - 목표 온도
            "humidity": 45.2,    # 실측 습도
            "weight": 4.65,      # 실측 무게
            "outer_temp": 22.1,
            "pressure": 1012.8,
            "fan_rpm": 1550.0,
            "user_action": 0
        },
        {
            "timestamp": 0.0,
            "inner_temp": 25,  # 실측 온도
            "error": -55,      # 실측 온도 - 목표 온도
            "humidity": 80,    # 실측 습도
            "weight": 5.0,     # 실측 무게
            "outer_temp": 22.0,
            "pressure": 1013.0,
            "fan_rpm": 1500.0,
            "user_action": 2
        }
    ]

    OVER_TEMP_LIMIT=15.0

    def __init__(self, reward_gen_fn_pathname=None, is_simulatation=True):
        """
        ; params reward_gen_fn: str, 보상 계산을 위한 사용자 정의 함수 (선택적)
        ' params is_simulation: boolean, 시뮬레이션 모드 여부 (True: 시뮬레이터, False: 실제 환경)
        """
        # super(DryerEnv, self).__init__()
        super().__init__()

        # 1. Action Space: [0: DOWN, 1: KEEP, 2: UP]
        self.action_space = spaces.Discrete(3)

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32
        )        
        self.is_simulation = is_simulatation

        self.target_temp = 80.0
        self.max_steps = 10000
        self.state = []
        self.terminated = False
        self.current_step = 0
        self.prev_inner_temp = None  # 이전 온도 저장용
        self.truncated = False
        self.reward_gen_fn_pathname: str = reward_gen_fn_pathname
        self.reward_gen_fn = None
        self.reset()
    
    def _generate_simul_data(self, state, action):
        timestamp, inner_temp, error, humidity, weight, outer_temp, pressure, fan_rpm = state
        # 습도 및 무게 자연 감소
        humidity = max(15.0, humidity - 0.01)
        weight = max(4.2, weight - 0.0001)

        # 1. 액션에 따른 팬 RPM 변화
        if action == 0:   # DOWN: 팬 속도 감소 -> 냉각 효율 저하 -> 온도 상승 유도
            fan_rpm -= 100
        elif action == 2: # UP: 팬 속도 증가 -> 냉각 효율 상승 -> 온도 하락 유도
            fan_rpm += 100
        
        fan_rpm = np.clip(fan_rpm, 0, 3000) # 가동 범위
    
        # 2. 열 평형 방정식 (핵심 로직)
        # 히터는 일정한 열을 공급 (Heater Input)
        heater_input = 10.0 
    
        # [핵심] 팬 RPM에 비례하는 냉각 계수 (Cooling Coefficient)
        # RPM이 커질수록 cooling_factor가 커져서 빼지는 값이 커짐
        cooling_factor = fan_rpm / 1500.0  # 기준 RPM 1500일 때 1.0
    
        # 외부 온도와의 차이에 의한 열 손실 (Heat Loss)
        # 팬이 빠를수록(cooling_factor ↑), 내부가 더울수록(temp diff ↑) 손실이 커짐
        heat_loss = cooling_factor * (inner_temp - outer_temp) * 0.1

        # 3. 최종 온도 변화
        # 공급(Heater)보다 손실(Loss)이 크면 온도는 떨어짐 (UP 액션 시 발생)
        # 공급(Heater)보다 손실(Loss)이 작으면 온도는 올라감 (DOWN 액션 시 발생)
        temp_change = heater_input - heat_loss
        new_inner_temp = inner_temp + temp_change

        # 4. 결과 검증 (인지 확인)
        # - UP(2) 클릭 -> RPM 상승 -> cooling_factor 상승 -> heat_loss 상승 -> temp_change 감소 -> 온도 하락
        # - DOWN(0) 클릭 -> RPM 하락 -> cooling_factor 하락 -> heat_loss 하락 -> temp_change 증가 -> 온도 상승
        print(f"Action: {action}, Fan RPM: {fan_rpm:.1f}, Cooling Factor: {cooling_factor:.3f}, Heat Loss: {heat_loss:.2f}, Temp Change: {temp_change:.2f}, New Inner Temp: {new_inner_temp:.2f}" )
        return {
                "timestamp": timestamp,
                "inner_temp": new_inner_temp,
                "error": new_inner_temp - self.target_temp,
                "humidity": humidity,
                "weight": weight,
                "outer_temp": outer_temp,
                "pressure": pressure,
                "fan_rpm": fan_rpm}

    def _send_action_to_dryer(self, action):
        """
        건조기 하드웨어로 제어 신호(RPM 조절)를 전송하는 함수
        실제 환경에서는 MQTT publish나 API POST 요청이 들어갑니다.
        """
        action_map = {0: "DOWN", 1: "KEEP", 2: "UP"}
        command = action_map[action]
        
        # 예시: requests.post(f"http://dryer-ip/control", json={"action": command})
        # print(f"[Hardware] Sent command: {command} to {self.device_id}")
        pass

    def _get_data_from_dryer(self, action):
        """
        건조기 센서로부터 실측 데이터를 읽어오는 함수
        실제 환경에서는 센서 API 호출이나 DB 조회가 들어갑니다.
        """
        # 예시: response = requests.get(f"http://dryer-ip/sensors").json()
        # 가상의 실측 데이터 구조 (API 응답 가정)
        if self.is_simulation:
            if len(self.state) == 0:
                real_measured_data = DryerEnv.init_sim_data[1]
            else:
                real_measured_data = self._generate_simul_data(self.state, action)
        else:
            ## 실제 상황에서는 건조기에서 데이터를 가져와야 합니다. 
            real_measured_data = DryerEnv.init_sim_data[0]
            
        return np.array([real_measured_data["timestamp"],
                         real_measured_data["inner_temp"],
                         real_measured_data["error"],
                         real_measured_data["humidity"],
                         real_measured_data["weight"],
                         real_measured_data["outer_temp"],
                         real_measured_data["pressure"],
                         real_measured_data["fan_rpm"]],
                         dtype=np.float32)

    def reset(self, seed=None, options=None, is_gen_data=True):
        super().reset(seed=seed)
        
        self.current_step = 0
        if self.terminated:
            pre_user_action = 0
        else:
            pre_user_action = 1
        if is_gen_data:
            self.state = self._get_data_from_dryer(action=pre_user_action)
        else:
            self.state = None
        self.terminated = False
        self.truncated = False
        self.reward_gen_fn = None
        self.prev_inner_temp = None
        return self.state, {}
    
    @staticmethod
    def _unpack_nptype(np_value):
        import numpy
        if isinstance(np_value, (numpy.float32, numpy.float64, np.float32, np.float64,
                                 numpy.int32, numpy.int64, np.int32, np.int64,
                                 numpy.ndarray, np.ndarray)):
            return np_value.item()
        return np_value

    def _generate_reward_default(self, current_temp, action):
        """
        # 보상(Reward)
        ; params inner_temp;
        ; return reward, terminated, truncated
        """
        diff = current_temp - self.target_temp
        reward = -abs(np.diff)
    
        # 올바른 대응에 보너스
        if np.diff < -1.0 and action == 0: reward += 20.0 # 추울 때 RPM 낮추면 칭찬
        if diff > 1.0 and action == 2:  reward += 20.0 # 더울 때 RPM 높이면 칭찬
        # 목표 온도 근처(±2도)에 도달 시 추가 보상
        if abs(diff) <= 1.0:
            reward += 30.0

        # if diff > 1.0: # 온도가 높음
        #     reward += (20.0 if action == 2 else -30.0) # UP이면 상, DOWN/KEEP이면 큰 벌

        # 4. 종료 조건
        terminated = self.current_step >= self.max_steps
        truncated = False
        # 온도가 너무 높거나 낮아지면 에피소드 조기 종료 (Safety)
        max_allow_temp = DryerEnv.OVER_TEMP_LIMIT + self.target_temp
        min_allow_temp = DryerEnv.OVER_TEMP_LIMIT
        if current_temp > max_allow_temp or current_temp < min_allow_temp:
            terminated = True
            reward -= 100
        return reward, terminated, truncated

    def _generate_reward(self, inner_temp, action):
        if self.reward_gen_fn_pathname is None or len(self.reward_gen_fn_pathname) == 0:
            reward = self._generate_reward_default(current_temp=inner_temp, action=action)
        else:
            from lib.module import load_function
            if self.reward_gen_fn_pathname is not None and len(self.reward_gen_fn_pathname) > 0:
                if self.reward_gen_fn is None:
                    self.reward_gen_fn = load_function(self.reward_gen_fn_pathname, "generate_reward")
                if self.reward_gen_fn:
                    reward = self.reward_gen_fn(inner_temp=inner_temp,
                                                action=action,
                                                prev_inner_temp=self.prev_inner_temp,
                                                target_temp=self.target_temp,
                                                temp_limit_depth=DryerEnv.OVER_TEMP_LIMIT)
                else:
                    print("Failed to load reward generation function.")
                    reward = -1000.0
            else:
                print("No reward generation function provided.")
                reward = -1000.0
        return reward
    
    def _check_terminate(self, inner_temp, reward):
        """
        4. 종료 조건
        """
        # 온도가 너무 높거나 낮아지면 에피소드 조기 종료 (Safety)
        if inner_temp > 90 or inner_temp < 15:
            self.terminated = True
            reward -= 100
        else:
            self.terminated = False
        if self.current_step > self.max_steps:
            self.terminated = True
        
        self.pre_user_action = 0 if self.terminated else action
        self.truncated = False
        return reward
    
    def _load_observation(self, action: int = 1, current_step: int = 0):
        # 실제 환경에서는 센서 데이터를 읽어와서 self.state를 업데이트해야 합니다.
        # 예시: self.state = self._get_data_from_dryer(action=self.pre_user_action)
        # print(f"Step {self.current_step}: Action taken: {action}")
        return self._get_data_from_dryer(action=action)

    def step(self, action):
        self.current_step += 1

        action = DryerEnv._unpack_nptype(action)

        # print(f'self.state #-1 = {self.state}, {action}[{type(action)}]')        
        self._send_action_to_dryer(action)

        if self.current_step > self.max_steps:
            self.current_step = self.max_steps # 인덱스 초과 방지

        # print(f'self.state #0 = {self.state}')
        self.state = self._load_observation()

        for i in range(len(self.state)):
            self.state[i] = DryerEnv._unpack_nptype(self.state[i])
        # print(f'self.state #1 = {self.state}')

        inner_temp_pos = 1
        inner_temp = self.state[inner_temp_pos]
        # 5. 보상 계산
        reward = self._generate_reward(inner_temp=inner_temp, action=action)

        # 4. 종료 조건
        reward = self._check_terminate(inner_temp=inner_temp, reward=reward)

        # 5. 다음 스텝을 위해 현재 온도를 이전 온도로 저장
        self.prev_inner_temp = inner_temp
        return self.state, reward, self.terminated, self.truncated, {}


# 환경 테스트
if __name__ == "__main__":
    env = DryerEnv(reward_gen_fn_pathname=reward_gen_fn_pythonfile, is_simulatation=True)
    obs, _ = env.reset()
    print(f"Initial Obs: {obs}")
    
    # 랜덤 액션 테스트
    for i in range(10):
        action = env.action_space.sample()
        in_temp = obs[1]
        obs, reward, done, _, _ = env.step(action)
        print(f"[{i:03d} Action: {action}, In Temp: {in_temp:.2f}, Out Temp: {obs[1]:.2f}, Reward: {reward:.2f}")

Action: 1, Fan RPM: 1500.0, Cooling Factor: 1.000, Heat Loss: 0.30, Temp Change: 9.70, New Inner Temp: 34.70
Initial Obs: [   0.       34.7     -45.3      79.99      4.9999   22.     1013.
 1500.    ]
Action: 1, Fan RPM: 1500.0, Cooling Factor: 1.000, Heat Loss: 1.27, Temp Change: 8.73, New Inner Temp: 43.43
[000 Action: 0, In Temp: 34.70, Out Temp: 43.43, Reward: -58.14
Action: 1, Fan RPM: 1500.0, Cooling Factor: 1.000, Heat Loss: 2.14, Temp Change: 7.86, New Inner Temp: 51.29
[001 Action: 1, In Temp: 43.43, Out Temp: 51.29, Reward: -82.43
Action: 1, Fan RPM: 1500.0, Cooling Factor: 1.000, Heat Loss: 2.93, Temp Change: 7.07, New Inner Temp: 58.36
[002 Action: 0, In Temp: 51.29, Out Temp: 58.36, Reward: -28.28
Action: 1, Fan RPM: 1500.0, Cooling Factor: 1.000, Heat Loss: 3.64, Temp Change: 6.36, New Inner Temp: 64.72
[003 Action: 1, In Temp: 58.36, Out Temp: 64.72, Reward: -55.56
Action: 1, Fan RPM: 1500.0, Cooling Factor: 1.000, Heat Loss: 4.27, Temp Change: 5.73, New Inner Temp: 70.4

#### Step 3 모방 학습(Behavioral Cloning)으로 PPO 초기화하기

##### Step 3.1 가상 데이터를 로딩하는 Env 작성

In [63]:
import gymnasium as gym
from gymnasium import spaces
import pandas as pd
import numpy as np

class CSVDataDryerEnv(DryerEnv):
    def __init__(self, csv_file, reward_gen_fn_pathname=None):
        # 데이터 로드
        self.df = pd.read_csv(csv_file)
        self.max_steps = len(self.df) - 1
        # super(CSVDataDryerEnv, self).__init__(reward_gen_fn_pathname=reward_gen_fn_pathname,
        #                                       is_simulatation=False)
        super().__init__(reward_gen_fn_pathname=reward_gen_fn_pathname, is_simulatation=False)

    def _load_observation(self, action: int = 1, current_step: int = 0):
        # CSV의 특정 행에서 데이터를 가져와서 state로 반환
        row = self.df.iloc[current_step]
        return np.array([
            row['timestamp'],
            row['inner_temp'],
            row['inner_temp'] - self.target_temp,
            row['humidity'],
            row['weight'],
            row['outer_temp'],
            row['pressure'],
            row['fan_rpm']
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed, is_gen_data=False)
        self.state = self._load_observation(action=1, current_step=self.current_step)
        return self.state, {}


##### 3.2 모방 학습 하기


In [64]:
import gymnasium as gym
from stable_baselines3 import PPO
# from lib.callback.ppo_event_callback import PPOEventCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor


cb_verbose = 1


def train_dryer_model_from_csv(model_path_name,
                               n_steps=128,
                               csv_path: str='dryer_virtual_data.csv',
                               reward_gen_fn_pathname=None,
                               learn_repeat_cnt: int=5):
    # 1. CSV 기반 환경 생성
    base_env = CSVDataDryerEnv(csv_file=csv_path,
                               reward_gen_fn_pathname=reward_gen_fn_pathname)
    env = DummyVecEnv([lambda: base_env])
    # callback = PPOEventCallback(check_freq=20, cb_verbose=cb_verbose)

    # 2. 모델 설정 (Linear 기반 PPO)
    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,
        n_steps=n_steps,
        batch_size=64,
        learning_rate=1e-5
    )

    # 3. 학습 시작
    # CSV 데이터의 크기에 맞춰 학습 횟수를 조절하세요.
    print(f"{csv_path} 데이터를 기반으로 학습을 시작합니다...")
    model.learn(total_timesteps=len(base_env.df) * learn_repeat_cnt,  # 데이터를 5번 반복 학습
                )

    trained_model_path_name = model_path_name()
    # 4. 저장
    model.save(trained_model_path_name)
    print("학습 및 모델 저장 완료")
    
    return model, trained_model_path_name

n_steps = 512
learn_repeat_cnt = 1
# 학습 실행
# checkpoint_callback = CheckpointCallback(save_freq=5000, save_path='./logs/', name_prefix='dryer_model')
# trained_model = train_dryer_model(model_path_name=model_path_name, total_timesteps=10000, n_epochs=1)
print(f"Trained will start. [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]")

trained_model, trained_model_path_name = train_dryer_model_from_csv(model_path_name=model_path_name,
                                                                    n_steps=n_steps,
                                                                    csv_path=csv_file_pathname,
                                                                    reward_gen_fn_pathname=reward_gen_fn_pythonfile,
                                                                    learn_repeat_cnt=learn_repeat_cnt)
print(f"Trained model saved at: {trained_model_path_name}")


Trained will start. [2026-03-29 23:18:27]
./data/drying_oven_sensor_v1.1.csv 데이터를 기반으로 학습을 시작합니다...
학습 및 모델 저장 완료
Trained model saved at: ./model/ppo_dryer_control_v1.1_20260329_231930.pth


##### 4. Test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
# from lib.callback.ppo_event_callback import PPOEventCallback
from stable_baselines3.common.vec_env import DummyVecEnv

inner_temp_history = []
action_history = []
reward_history = []
repeat_cnt = 20
learn_timesteps = 1
trained_model_path_name ="./model/ppo_dryer_control_v1.1_20260329_231930.pth"  # 방금 저장한 모델 경로 사용

def test_model(model_path_name, repeat_cnt=100, reward_gen_fn_pathname=None):
    # 저장된 모델 로드 및 테스트

    # 1. 원본 환경 생성
    base_env = DryerEnv(reward_gen_fn_pathname=reward_gen_fn_pathname)
    env = DummyVecEnv([lambda: base_env])
    model = PPO.load(model_path_name)
    # 2. Monitor 래핑 (표준 규격 및 로그 기록)
    model.env = env
    obs = model.env.reset()

    in_obs = obs
    print(f"in_obs = {in_obs}")

    inner_temp_history.append(in_obs[0][1])
    
    for i in range(repeat_cnt):
        # 모델 예측 (deterministic=True: 가장 확률 높은 액션 선택)
        print(f"Step {i}: In ={in_obs}[{type(in_obs)}]")
        action, _ = model.predict(
            in_obs, 
            deterministic=True
        )
        step_results = model.env.step(action)
        print(f'step_results = {step_results}, {type(step_results)}')
        predict_obs = step_results[0][0].tolist()
        in_obs = [predict_obs]
        reward = step_results[1][0]
        print(f"Step {i}: In ={predict_obs}, Action={action}, Reward={reward:.2f}")
        if i > 0 and i % learn_timesteps == 0:
            print(f"Step {i}: Learning from experience...")
            model.learn(total_timesteps=learn_timesteps, reset_num_timesteps=False, log_interval=None)
        inner_temp_history.append(predict_obs[1])
        action_history.append(int(action.item()))
        reward_history.append(reward)
        
        # if terminated or truncated:
        #     break


print(f"Testing model {trained_model_path_name} for {repeat_cnt} steps...")

test_model(model_path_name=trained_model_path_name,
           repeat_cnt=1000,
           reward_gen_fn_pathname=reward_gen_fn_pythonfile)

plt.plot(inner_temp_history, label='Inner Temperature')
plt.axhline(15, color='r', linestyle='--', label='Target')
plt.title("Inner Temperature")
plt.legend()
plt.show()

Testing model ./model/ppo_dryer_control_v1.1_20260329_231930.pth for 100 steps...


NameError: name 'DummyVecEnv' is not defined